# Dataset Variant Comparison Experiments

Which dataset processing yields the best re-identification performance?

**Tested Dataset Variants:**
1. **Master**: Segmented + Deduplicated (recommended)
2. **Segmented Deduplicated**: Segmented + Deduplicated
3. **Segmented**: Segmented only (with duplicates)
4. **Not Segmented Deduplicated**: Full frames, deduplicated
5. **Not Segmented**: Full frames with duplicates

**Fixed Settings:**
- Backbone: MegaDescriptor-L-384
- Loss: ArcFace (m=0.5, s=64)
- Epochs: 50

Results logged to Wandb project: `jaguar-reid`, group: `dataset_comparison`

## Setup

In [ ]:
import sys
from pathlib import Path
import logging

# Add src to path
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import get_default_config
from jaguars.reidentification.experiments import get_dataset_experiments
from jaguars.reidentification.training.train import run_processing as run_training

logger = setup_logger("dataset_experiments", level=logging.INFO)
print("✓ Imports successful")

## Configuration

Configure base settings for all dataset experiments.

In [ ]:
# Get default configuration
config = get_default_config()

# Wandb settings
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"

# Backbone settings (fixed)
config.backbone.name = "BVRA/MegaDescriptor-L-384"
config.backbone.pretrained = True
config.backbone.embedding_dim = 1536

# Loss settings (fixed)
config.training.num_epochs = 50
config.training.loss_name = "arcface"
config.model.arcface_margin = 0.5
config.model.arcface_scale = 64.0

# FiftyOne common settings
config.dataset.source = "fiftyone"
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_label_field = "ground_truth"

print(f"✓ Base config loaded")
print(f"  Wandb project: {config.wandb.project}")
print(f"  Backbone: {config.backbone.name}")
print(f"  Loss: {config.training.loss_name}")
print(f"  Epochs: {config.training.num_epochs}")

## Load Experiments

In [ ]:
# Get dataset experiments with our custom config
dataset_experiments = get_dataset_experiments(base_config=config)

print(f"✓ {len(dataset_experiments)} dataset experiments configured:")
for exp in dataset_experiments:
    print(f"  - {exp.name}: {exp.description}")
    print(f"    Dataset: {exp.base_config.dataset.fo_dataset_name}")
    print(f"    Patches: {exp.base_config.dataset.fo_patches_field}")
    print(f"    Embeddings: {exp.base_config.dataset.fo_embeddings_field}")

## Run Experiments

Train with fixed backbone and loss on different dataset variants.

In [ ]:
# Run all dataset experiments
results = {}

for experiment in dataset_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Dataset: {experiment.base_config.dataset.fo_dataset_name}")
    logger.info(f"  Tags: {experiment.tags}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        # Run training
        result = run_training(experiment.base_config)
        results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(dataset_experiments)} dataset experiments completed")

## Results Summary

Compare dataset variants on validation set.

In [ ]:
# Print summary of results
import pandas as pd

summary_data = []
for exp_name, result in results.items():
    if "error" in result:
        summary_data.append({
            "Experiment": exp_name,
            "Validation mAP": "ERROR",
            "Status": result["error"]
        })
    else:
        summary_data.append({
            "Experiment": exp_name,
            "Validation mAP": result.get("validation/map", "N/A"),
            "Status": "✓ Completed"
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Dataset Comparison Results ===\n")
print(summary_df.to_string(index=False))
print(f"\nView detailed results at: https://wandb.ai/{config.wandb.entity}/{config.wandb.project}")